In [3]:
import torch
from denoising_diffusion_pytorch import Unet, GaussianDiffusion, Trainer1D, Dataset1D
from pathlib import Path
import numpy as np

/global/homes/k/kp22/.conda/envs/denoising_diffusion_pytorch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.get_device_name(0)

'NVIDIA A100-PCIE-40GB'

In [ ]:
model = Unet(
    dim = 64,
    dim_mults = (1, 2, 4, 8),
    channels=1,
    flash_attn = True
)

diffusion = GaussianDiffusion(
    model,
    image_size = 256,
    timesteps = 1000    # number of steps
)
diffusion.to(device);

In [6]:
ptsrc=2
fpath_cib = f"data/low_pass/{ptsrc}mJy/CIB_map_150GHz_256_st6_minmax_{ptsrc}mJy_zero_lp.npy"
fpath_tsz = f"data/low_pass/{ptsrc}mJy/tSZ_map_150GHz_256_st6_minmax_{ptsrc}mJy_zero_lp.npy"

cib_maps = np.load(fpath_cib)  # (N, H, W, 1)
tsz_maps = np.load(fpath_tsz)  # (N, H, W, 1)

# Stack along channel axis: (N, H, W, 2)
joint_maps = np.concatenate([cib_maps, tsz_maps], axis=-1)
joint_maps = joint_maps.transpose(0, 3, 1, 2)  # (N, 2, H, W)

In [7]:
joint_maps.shape

(1041, 2, 256, 256)

In [6]:
fpath = "data/low_pass/6mJy/cut_maps_RES_256_ANG_X_6.0 deg_6mJy_lp.npy"

cut_maps = np.load(fpath)
cut_maps = cut_maps.transpose(0, 3, 1, 2)

num_samples = len(cut_maps)
num_train = int(0.8 * num_samples)

# Create a local random number generator
rng = np.random.default_rng(seed=42)

# Shuffle and split the data
indices = rng.permutation(num_samples)
train_indices = indices[:num_train]

# Create the training tensor
training_images = torch.tensor(cut_maps[train_indices], dtype=torch.float32)

In [7]:
def augment_images_unique(training_images):
    augmented_images = []
    for image in training_images:
            transforms = [
                image,
                torch.rot90(image, k=1, dims=(1, 2)),   # 90°
                torch.rot90(image, k=2, dims=(1, 2)),   # 180°
                torch.rot90(image, k=3, dims=(1, 2)),   # 270°
            ]

            for t_img in transforms:
                augmented_images.append(t_img)  # original and rotated versions
                augmented_images.append(torch.flip(t_img, dims=[2]))  # horizontal flip of each rotated version

    return torch.stack(augmented_images)

In [8]:
augmented_images = augment_images_unique(training_images)

In [10]:
np.shape(augmented_images),np.shape(training_images)

(torch.Size([6656, 1, 256, 256]), torch.Size([832, 1, 256, 256]))

In [ ]:
total_params = sum(param.numel() for param in model.parameters())
print(f"Total parameters: {total_params/1e6} M")

In [ ]:
diffusion.objective

In [ ]:
from torchviz import make_dot
import graphviz
from torchview import draw_graph

In [ ]:
BATCH_SIZE, IMG_CH, IMG_SIZE = 32, 1, 256
timesteps =1000
dummy_input = (
    torch.randn(BATCH_SIZE, IMG_CH, IMG_SIZE, IMG_SIZE),
    torch.randint(0, timesteps, (BATCH_SIZE,))
)

graphviz.set_jupyter_format('png')
model_graph = draw_graph(
    model,
    input_data=dummy_input,
    device='meta',
    expand_nested=True
)
model_graph.resize_graph(scale=1.5)
model_graph.visual_graph
